In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as tv_models
from torch.optim.lr_scheduler import ReduceLROnPlateau
import numpy as np, os, random, time, copy, gc
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, classification_report)
from sklearn.model_selection import train_test_split
import wandb

# ===================== CONFIG =====================
# Vision Transformer (ViT) for Lung Cancer Classification
# Paper reference: Multiple 2024-2025 papers using ViT on IQ-OTH/NCCD
# Claimed: 95-99%+ accuracy with pretrained ViT
# THIS FILE: 80/20 image-level random split (leaky — paper method)
# ==================================================

config = {
    'data_dir': '/kaggle/input/datasets/hamdallak/the-iqothnccd-lung-cancer-dataset/The IQ-OTHNCCD lung cancer dataset',
    'batch_size': 16,             # ViT is memory-hungry
    'learning_rate': 1e-4,        # Lower LR for pretrained ViT
    'epochs': 30,
    'dropout': 0.3,
    'lr_decay_factor': 0.3,
    'lr_patience': 5,
    'test_size': 0.20,
    'image_size': 224,            # ViT-B/16 expects 224x224
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'seed': 42,
    'class_names': ['Benign', 'Malignant', 'Normal'],
    'aug_factor_benign': 13,
    'aug_factor_malignant': 2,
    'aug_factor_normal': 3,
    'variants': ['vit_b_16'],  # B/32 too coarse (55%), L/16 too slow (341s/epoch)
}

# ===================== REPRODUCIBILITY =====================
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed(config['seed'])

# ===================== DATA LOADING =====================
CLASS_MAP = {"Bengin cases": 0, "Malignant cases": 1, "Normal cases": 2}

def collect_all_images(data_dir):
    all_samples = []
    for cls_name, cls_id in CLASS_MAP.items():
        cls_dir = os.path.join(data_dir, cls_name)
        if not os.path.exists(cls_dir):
            for split in ['train', 'test', 'valid']:
                split_class = os.path.join(data_dir, split, cls_name)
                if os.path.exists(split_class):
                    cls_dir = split_class; break
        if os.path.exists(cls_dir):
            files = sorted([f for f in os.listdir(cls_dir)
                          if f.lower().endswith(('.jpg','.jpeg','.png','.bmp'))])
            for f in files:
                all_samples.append((os.path.join(cls_dir, f), cls_id))
    return all_samples

def crop_lung_contour(image_np):
    gray = cv2.cvtColor(image_np, cv2.COLOR_RGB2GRAY) if len(image_np.shape)==3 else image_np.copy()
    _, thresh = cv2.threshold(gray, 20, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        largest = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(largest)
        pad = 5; x, y = max(0, x-pad), max(0, y-pad)
        w = min(image_np.shape[1]-x, w+2*pad); h = min(image_np.shape[0]-y, h+2*pad)
        return image_np[y:y+h, x:x+w]
    return image_np

class LungCropTransform:
    def __call__(self, image):
        return Image.fromarray(crop_lung_contour(np.array(image)))

# ===================== DATASET =====================
class LungDataset(Dataset):
    def __init__(self, samples, transform, oversample_factors=None):
        self.transform = transform
        if oversample_factors:
            self.samples = []
            for path, label in samples:
                self.samples.extend([(path, label)] * oversample_factors.get(label, 1))
        else:
            self.samples = list(samples)
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        return self.transform(Image.open(path).convert('RGB')), label

# ===================== TRANSFORMS =====================
def get_transforms(image_size):
    train_tf = transforms.Compose([
        LungCropTransform(),
        transforms.Resize((image_size, image_size)),
        transforms.RandomHorizontalFlip(0.5),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.2),
        transforms.RandomAffine(0, translate=(0.1,0.1), scale=(0.9,1.1)),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
    ])
    eval_tf = transforms.Compose([
        LungCropTransform(),
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
    ])
    return train_tf, eval_tf

# ===================== DATA SPLIT (LEAKY) =====================
def prepare_paper_data(data_dir, test_size=0.20, batch_size=16, image_size=224):
    """80/20 image-level random split (leaky — as papers describe)."""
    all_samples = collect_all_images(data_dir)
    labels = [s[1] for s in all_samples]
    print(f"Total images: {len(all_samples)}")
    for name, cid in CLASS_MAP.items():
        print(f"  {name}: {sum(1 for _,l in all_samples if l==cid)}")

    train_samples, test_samples = train_test_split(
        all_samples, test_size=test_size, random_state=config['seed'], stratify=labels)
    train_labels = [s[1] for s in train_samples]
    train_samples, val_samples = train_test_split(
        train_samples, test_size=0.10, random_state=config['seed'], stratify=train_labels)
    print(f"\nAfter 80/20 split: train={len(train_samples)}, val={len(val_samples)}, test={len(test_samples)}")

    train_tf, eval_tf = get_transforms(image_size)
    aug_factors = {0: config['aug_factor_benign'], 1: config['aug_factor_malignant'],
                   2: config['aug_factor_normal']}
    train_ds = LungDataset(train_samples, train_tf, oversample_factors=aug_factors)
    val_ds = LungDataset(val_samples, eval_tf)
    test_ds = LungDataset(test_samples, eval_tf)
    print(f"Train after augmentation: {len(train_ds)}")

    kw = dict(num_workers=4, pin_memory=True, persistent_workers=True, prefetch_factor=4)
    return (DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True, **kw),
            DataLoader(val_ds, batch_size=batch_size, shuffle=False, **kw),
            DataLoader(test_ds, batch_size=batch_size, shuffle=False, **kw))

# ===================== MODEL =====================
def build_vit(variant='vit_b_16', num_classes=3, dropout=0.3, pretrained=True):
    builders = {
        'vit_b_16': (tv_models.vit_b_16, tv_models.ViT_B_16_Weights.IMAGENET1K_V1),
        'vit_b_32': (tv_models.vit_b_32, tv_models.ViT_B_32_Weights.IMAGENET1K_V1),
        'vit_l_16': (tv_models.vit_l_16, tv_models.ViT_L_16_Weights.IMAGENET1K_V1),
    }
    model_fn, weights = builders[variant]
    model = model_fn(weights=weights if pretrained else None)
    in_features = model.heads.head.in_features
    model.heads.head = nn.Sequential(nn.Dropout(p=dropout), nn.Linear(in_features, num_classes))
    total = sum(p.numel() for p in model.parameters())
    print(f"  {variant}: {total/1e6:.2f}M params")
    return model

# ===================== TRAINING =====================
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    loss_sum, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device, non_blocking=True), (torch.tensor(labels, device=device) if not isinstance(labels, torch.Tensor) else labels.to(device))
        optimizer.zero_grad()
        out = model(imgs); loss = criterion(out, labels)
        loss.backward(); optimizer.step()
        loss_sum += loss.item()*imgs.size(0)
        correct += (out.argmax(1)==labels).sum().item(); total += imgs.size(0)
    return loss_sum/total, correct/total

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    all_preds, all_labels, all_probs = [], [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(device, non_blocking=True), (torch.tensor(labels, device=device) if not isinstance(labels, torch.Tensor) else labels.to(device))
        out = model(imgs); loss = criterion(out, labels)
        loss_sum += loss.item()*imgs.size(0)
        correct += (out.argmax(1)==labels).sum().item(); total += imgs.size(0)
        all_preds.extend(out.argmax(1).cpu().tolist())
        all_labels.extend(labels.cpu().tolist())
        all_probs.extend(F.softmax(out, dim=1).cpu().tolist())
    return loss_sum/total, correct/total, all_preds, all_labels, all_probs

def train_full(model, train_ld, val_ld, test_ld, name, device, epochs=30, lr=1e-4):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=config['lr_decay_factor'],
                                  patience=config['lr_patience'])
    best_val_acc, best_wts, no_improve = 0, None, 0
    early_stop_patience = 7

    print(f"\n{'='*60}\n Training: {name} | {epochs} epochs | LR={lr}\n{'='*60}")
    for epoch in range(epochs):
        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch(model, train_ld, optimizer, criterion, device)
        vl_loss, vl_acc, vl_preds, vl_labels, _ = evaluate(model, val_ld, criterion, device)
        vl_f1 = f1_score(vl_labels, vl_preds, average='macro', zero_division=0)
        scheduler.step(vl_acc); cur_lr = optimizer.param_groups[0]['lr']

        marker = ''
        if vl_acc > best_val_acc:
            best_val_acc = vl_acc; best_wts = copy.deepcopy(model.state_dict())
            no_improve = 0; marker = ' ★'
        else:
            no_improve += 1

        print(f"  [{epoch+1:3d}/{epochs}] loss={tr_loss:.4f}/{vl_loss:.4f} "
              f"acc={tr_acc:.4f}/{vl_acc:.4f} f1={vl_f1:.4f} "
              f"lr={cur_lr:.1e} ({time.time()-t0:.1f}s){marker}")
        wandb.log({"epoch": epoch+1, "train/loss": tr_loss, "train/acc": tr_acc*100,
                    "val/loss": vl_loss, "val/acc": vl_acc*100, "val/f1": vl_f1*100,
                    "best_val_acc": best_val_acc*100, "learning_rate": cur_lr})

        if no_improve >= early_stop_patience:
            print(f"  ⏹ Early stop at epoch {epoch+1}"); break

    if best_wts: model.load_state_dict(best_wts)
    _, test_acc, test_preds, test_labels, test_probs = evaluate(model, test_ld, criterion, device)
    return model, {'name': name, 'test_acc': test_acc, 'test_preds': test_preds,
                   'test_labels': test_labels, 'test_probs': test_probs,
                   'params': sum(p.numel() for p in model.parameters())}

# ===================== EVALUATION =====================
def full_evaluation(results, class_names):
    preds, labels, probs = results['test_preds'], results['test_labels'], results['test_probs']
    print(f"\n{'='*60}\n TEST RESULTS — {results['name']}\n{'='*60}")
    print(f"Accuracy:  {results['test_acc']*100:.2f}%")
    print(f"Parameters: {results['params']/1e6:.2f}M\n")
    print(classification_report(labels, preds, target_names=class_names, zero_division=0))

    cm = confusion_matrix(labels, preds)
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names,
                yticklabels=class_names, ax=ax, annot_kws={'size': 16})
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(f"{results['name']} — Confusion Matrix"); plt.tight_layout()
    wandb.log({f"confusion/{results['name']}": wandb.Image(fig)})
    plt.show(); plt.close()

    try: auc_val = roc_auc_score(labels, probs, multi_class='ovr', average='macro')
    except: auc_val = 0
    wandb.log({f"test/{results['name']}_acc": results['test_acc']*100,
               f"test/{results['name']}_auc": auc_val*100})

# ===================== XAI: Attention Rollout =====================
def get_vit_attention(model, image_tensor, device):
    """Extract ViT self-attention maps from all layers."""
    model.eval()
    attentions = []
    hooks = []
    for block in model.encoder.layers:
        def hook_fn(module, inp, out, attn_store=attentions):
            # ViT encoder block's self_attention
            attn_store.append(out[1] if isinstance(out, tuple) else None)
        hooks.append(block.self_attention.register_forward_hook(hook_fn))

    with torch.no_grad():
        _ = model(image_tensor.unsqueeze(0).to(device))

    for h in hooks: h.remove()
    return attentions

def visualize_xai(model, test_loader, device):
    """Grad-CAM alternative: use ViT's encoder block attention for visualization."""
    model.eval()
    samples = {}
    for imgs, labels in test_loader:
        for i in range(imgs.size(0)):
            lbl = labels[i].item() if isinstance(labels[i], torch.Tensor) else labels[i]
            if lbl not in samples: samples[lbl] = imgs[i]
        if len(samples) >= 3: break

    # Simple Grad-CAM on last encoder block's layer norm
    fig, axes = plt.subplots(3, 3, figsize=(14, 12))
    for row, (lbl, img_t) in enumerate(sorted(samples.items())):
        img_np = img_t.permute(1,2,0).cpu().numpy()
        img_np = img_np * np.array([0.229,0.224,0.225]) + np.array([0.485,0.456,0.406])
        img_np = np.clip(img_np, 0, 1)

        # Get activation from last encoder layer
        activations = {}
        def fwd_hook(m, i, o): activations['v'] = o.detach()
        gradients = {}
        def bwd_hook(m, gi, go): gradients['v'] = go[0].detach()

        target = model.encoder.layers[-1].ln_1
        h1 = target.register_forward_hook(fwd_hook)
        h2 = target.register_full_backward_hook(bwd_hook)

        inp = img_t.unsqueeze(0).to(device).requires_grad_(True)
        out = model(inp); model.zero_grad()
        out[0, lbl].backward()
        h1.remove(); h2.remove()

        # Reshape patch tokens to spatial grid (exclude CLS token)
        act = activations['v'][0, 1:]  # (196, 768) for ViT-B/16
        grad = gradients['v'][0, 1:]
        n_patches = int(act.shape[0] ** 0.5)
        weights = grad.mean(dim=1)  # (196,)
        cam = weights.view(n_patches, n_patches).cpu().numpy()
        cam = np.maximum(cam, 0)
        cam = cv2.resize(cam, (config['image_size'], config['image_size']))
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

        axes[row,0].imshow(img_np); axes[row,0].set_title(f'Original\nTrue: {config["class_names"][lbl]}'); axes[row,0].axis('off')
        axes[row,1].imshow(cam, cmap='jet'); axes[row,1].set_title('Attention Map'); axes[row,1].axis('off')
        overlay = np.clip(0.5*img_np + 0.5*plt.cm.jet(cam)[:,:,:3], 0, 1)
        axes[row,2].imshow(overlay); axes[row,2].set_title('Overlay'); axes[row,2].axis('off')

    fig.suptitle('XAI: ViT Attention Visualization (Last Encoder Block)', fontsize=15, fontweight='bold')
    plt.tight_layout()
    wandb.log({"xai/attention": wandb.Image(fig)})
    plt.show(); plt.close()
    print("\n✅ XAI visualizations generated.")

# ===================== MAIN =====================
if __name__ == '__main__':
    from kaggle_secrets import UserSecretsClient
    wandb.login(key=UserSecretsClient().get_secret("wandb_api_key"))

    wandb.init(project="vit_lung_reproduction", name="vit_leaky_split", config=config,
               tags=["paper-method", "leaky", "80-20", "vit", "transformer"],
               notes="ViT lung cancer: 80/20 image-level split (leaky)")

    print("="*60)
    print("ViT Lung Cancer — Image-Level 80/20 Split (Leaky)")
    print("="*60)
    train_ld, val_ld, test_ld = prepare_paper_data(
        config['data_dir'], test_size=config['test_size'],
        batch_size=config['batch_size'], image_size=config['image_size'])

    wandb.log({"dataset/train_samples": len(train_ld.dataset),
               "dataset/val_samples": len(val_ld.dataset),
               "dataset/test_samples": len(test_ld.dataset),
               "dataset/split_method": "image-level random (leaky)"})

    all_results = {}
    best_model, best_acc = None, 0

    for variant in config['variants']:
        print(f"\n{'#'*60}")
        print(f" {variant} | LEAKY SPLIT")
        print(f"{'#'*60}")
        try:
            model = build_vit(variant, dropout=config['dropout'])
            model, results = train_full(model, train_ld, val_ld, test_ld,
                                         variant, config['device'],
                                         epochs=config['epochs'], lr=config['learning_rate'])
            full_evaluation(results, config['class_names'])
            all_results[variant] = results
            if results['test_acc'] > best_acc:
                best_acc = results['test_acc']; best_model = model
        except RuntimeError as e:
            if 'out of memory' in str(e).lower():
                print(f"  ⚠️ {variant} OOM — skipping (needs more GPU RAM)")
                torch.cuda.empty_cache(); gc.collect()
            else:
                raise
        del model; torch.cuda.empty_cache(); gc.collect()

    # Summary
    print(f"\n{'='*60}")
    print(f" SUMMARY: ViT Variants (Leaky Split)")
    print(f"{'='*60}")
    for v, r in all_results.items():
        print(f"  {v}: {r['test_acc']*100:.2f}%")

    if best_model:
        print("\nGenerating XAI on best model...")
        visualize_xai(best_model, test_ld, config['device'])

    wandb.finish()
    print("Done!")
